### 2.2 환경 변수 설정

In [37]:
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

True

### 2.3 기본 라이브러리

In [38]:
import os
import json
from glob import glob
from pprint import pprint
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns

# # 한글 폰트 인식 - Windows
# import matplotlib 
# font_name = matplotlib.font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
# matplotlib.rc('font', family=font_name)

# 한글 폰트 인식 - Mac
import matplotlib
matplotlib.rc('font', family='AppleGothic')

# 마이너스 부호 인식
matplotlib.rc("axes", unicode_minus = False)

# LangChain 핵심
from langchain_core.documents import Document
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 데이터 처리
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 평가
import ranx_k

단계 1

In [39]:
def load_text_files(file_patterns):

    documents = []
    
    for pattern in file_patterns:
        files = glob(pattern)
        for file_path in files:
            try:
                loader = TextLoader(file_path, encoding='utf-8')
                docs = loader.load()
                documents.extend(docs)
                print(f"✅ {file_path} 로드 완료")
            except Exception as e:
                print(f"❌ {file_path} 로드 실패: {e}")
    
    return documents

# 로드할 문서의 패턴 정의
# 예시: 'data/*.txt', 'data/*.json' 등
# 현재는 'data/*_KR.md' 패턴만 사용


### 채널 오류 코드 문서 분할 및 전처리

In [ ]:
file_patterns = [
    'data/prj_*.csv',
]
raw_documents = load_text_files(file_patterns)
print(f"총 {len(raw_documents)}개 문서 로드됨")

def preprocess_documents(documents, error_mapping=None):
    """
    문서 전처리 및 메타데이터 추가
    
    Args:
        documents (list): 원본 문서 리스트
        error_mapping (dict): 오류명 매핑 정보
    
    Returns:
        list: 전처리된 Document 객체 리스트
    """
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base",
        separators=['\n\n', '\n', r'(?<=[.!?])\s+'],
        chunk_size=150,
        chunk_overlap=50,
        is_separator_regex=True,
        keep_separator=True,
    )
    chunks = text_splitter.split_documents(documents)
    
    processed_docs = []
    for chunk in chunks:
        # 메타데이터 추가
        #source_file = chunk.metadata.get('코드',).get('source', '')

        source_file = chunk.page_content[]        
        
        # 회사명 추출
        if error_mapping:
            error = 'Unknown'
            for keyword, name in error_mapping.items():
                print(source_file.lower(), "-----> ", keyword, name)
                if keyword in source_file.lower():
                    error = name
                    break
        else:
            error = 'default'
        
        # Document 객체 생성
        doc = Document(
            page_content=f"<Document>\n{chunk.page_content}\n</Document>\n<Source>이 문서는 '{error}'에 대한 문서입니다.</Source>",
            metadata={
                **chunk.metadata,
                'error': error,
                'language': 'ko',
                'chunk_length': len(chunk.page_content)
            }
        )
        processed_docs.append(doc)
    
    return processed_docs

# 오류 메시지 목록
error_msg = {
    'Bad Request':'400',
    'Unauthorized':'401',
    'Forbidden':'403',
    'Not Found':'404',
    'Internal Server Error':'500',
}

processed_docs = preprocess_documents(raw_documents, error_msg)
print(f"총 {len(processed_docs)}개 청크 생성됨")

# 결과 확인
for i, doc in enumerate(processed_docs):
    print(f"\n[청크 {i+1}]")
    print(f"오류: {doc.metadata['error']}")
    print(f"내용: {doc.page_content}")
    print("-" * 50)




✅ data/prj_chn_error_lists.csv 로드 완료
총 1개 문서 로드됨
﻿http 상태 코드,http 상태 메시지,http 응답 바디,발생원인,해결방안
403,forbidden,"{
  ""message"" : ""forbidden"",
  ""status"" : 403
}",api key 누락,"http 헤더에 “x-api-key” 필드를 추가하고 발급받은 api key를 값으로 사용하여 요청 전달하도록 수정 ----->  Bad Request 400
﻿http 상태 코드,http 상태 메시지,http 응답 바디,발생원인,해결방안
403,forbidden,"{
  ""message"" : ""forbidden"",
  ""status"" : 403
}",api key 누락,"http 헤더에 “x-api-key” 필드를 추가하고 발급받은 api key를 값으로 사용하여 요청 전달하도록 수정 ----->  Unauthorized 401
﻿http 상태 코드,http 상태 메시지,http 응답 바디,발생원인,해결방안
403,forbidden,"{
  ""message"" : ""forbidden"",
  ""status"" : 403
}",api key 누락,"http 헤더에 “x-api-key” 필드를 추가하고 발급받은 api key를 값으로 사용하여 요청 전달하도록 수정 ----->  Forbidden 403
﻿http 상태 코드,http 상태 메시지,http 응답 바디,발생원인,해결방안
403,forbidden,"{
  ""message"" : ""forbidden"",
  ""status"" : 403
}",api key 누락,"http 헤더에 “x-api-key” 필드를 추가하고 발급받은 api key를 값으로 사용하여 요청 전달하도록 수정 ----->  Not Found 404
﻿http 상태 코드,http 상태 메시지,http 응답 바디,발생원인,해결방안
403,forbidden,"{
  ""message"" : ""forbidden"

In [ ]:
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

def create_or_load_vector_store(documents, collection_name="prj_chn_error_lists_db"):
    """
    Chroma 벡터 저장소 생성 (기존 컬렉션이 있으면 로드하고, 없으면 새로 생성)
    
    Args:
        documents (list): Document 객체 리스트
        collection_name (str): 컬렉션 이름
    
    Returns:
        Chroma: 벡터 저장소 객체

    """
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    persist_directory = "./local_chroma_db"
    
    # 디렉토리와 컬렉션 파일이 존재하는지 확인
    if os.path.exists(persist_directory):
        try:
            # 기존 컬렉션 로드 시도
            vector_store = Chroma(
                collection_name=collection_name,
                embedding_function=embeddings,
                persist_directory=persist_directory
            )
            
            # 컬렉션에 문서가 있는지 확인
            if vector_store._collection.count() > 0:
                print(f"✅ 기존 벡터 저장소 로드 완료: {vector_store._collection.count()}개 문서")
                return vector_store
        except Exception as e:
            print(f"기존 컬렉션 로드 실패: {e}")
    
    # 새로운 벡터 저장소 생성
    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=persist_directory,
        collection_metadata={'hnsw:space': 'cosine'}
    )
    print(f"✅ 새 벡터 저장소 생성 완료: {vector_store._collection.count()}개 문서")
    return vector_store

# 벡터 저장소 생성
vector_store = create_or_load_vector_store(processed_docs)

---

## 4. 검색 방법론

### 4.1 의미론적 검색 (Semantic Search)

#### 4.1.1 벡터 저장소 생성

#### 4.1.2 의미론적 검색 실행

In [47]:
def semantic_search(vector_store, query, k=5):
    """
    의미론적 검색 실행
    
    Args:
        vector_store: Chroma 벡터 저장소
        query (str): 검색 쿼리
        k (int): 반환할 문서 수
    
    Returns:
        list: 검색된 Document 객체 리스트
    """
    # 검색기 생성
    retriever = vector_store.as_retriever(search_kwargs={"k": k})
    
    # 검색 실행
    results = retriever.invoke(query)
        
    return results

# 검색 테스트
query = "Forbidden 관련 오류는?"
semantic_results = semantic_search(vector_store, query, k=3)

print(f"Query: {query}")
print(f"검색 결과 ({len(semantic_results)}개)")
for i, doc in enumerate(semantic_results, 1):
    print(f"\n[{i}] 오류: {doc.metadata.get('error', 'N/A')}")
    print(f"내용:\n{doc.page_content}")
    print("-" * 50)


Query: Forbidden 관련 오류는?
검색 결과 (3개)

[1] 오류: unknown
내용:
<Document>
403,Forbidden,"{
  ""message"" : ""Access Restricted"",
  ""status"" : 403
}",서버 IP/Referer Whitelist 모두 기재,“IP/Referer Whitelist” 속성에 IP와 Referer가 모두 등록된 경우 제휴사 정책에 따라 한 쪽은 삭제
403,Forbidden,"{
  ""error"" : ""API 접근 권한이 없습니다."",
  ""message"" : ""접근이 허가되지 않은 API 호출입니다."",
</Document>
<Source>이 문서는 'unknown'에 대한 문서입니다.</Source>
--------------------------------------------------

[2] 오류: unknown
내용:
<Document>
""error"" : ""API 접근 권한이 없습니다."",
  ""message"" : ""접근이 허가되지 않은 API 호출입니다."",
  ""timestamp"" : ""20241104150507"",
  ""status"" : 403
}",HTTP 메소드를 오지정,"HTTP 호출시 메소드를 정확하게 세팅하였는지 확인
API마다 서비스 특성에 따라 GET, POST로 메소드가 지정되어 있음 "
403,Forbidden,"{
</Document>
<Source>이 문서는 'unknown'에 대한 문서입니다.</Source>
--------------------------------------------------

[3] 오류: unknown
내용:
<Document>
""error"": ""token not authorized"", 
   ""message"": ""token not authorized"", 
   ""timestamp"": ""20241105102919"", 
   ""status"": 401

### 4.2 키워드 검색 (Keyword Search)

#### 4.2.1 한국어 토크나이저 설정

In [11]:
def setup_korean_tokenizer():
    """
    한국어 토크나이저 설정
    
    Returns:
        Kiwi: 한국어 토크나이저 객체
    """
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    
    # 사용자 정의 단어 추가
    custom_words = [
        ('리비안', 'NNP'),  # 고유명사
        ('테슬라', 'NNP'),  # 고유명사
        ('전기차', 'NNG'),  # 일반명사
    ]
    
    for word, pos in custom_words:
        kiwi.add_user_word(word, pos)
        print(f"✅ 단어 추가: {word} ({pos})")
    
    return kiwi

def korean_tokenizer(text, kiwi_model):
    """
    한국어 토크나이저 함수
    
    Args:
        text (str): 토큰화할 텍스트
        kiwi_model: Kiwi 모델 객체
    
    Returns:
        list: 토큰 리스트
    """
    return [token.form for token in kiwi_model.tokenize(text)]

# 토크나이저 설정
kiwi_model = setup_korean_tokenizer()

# 토큰화 테스트
test_text = "리비안은 언제 설립되었나요?"
tokens = korean_tokenizer(test_text, kiwi_model)
print(f"원문: {test_text}")
print(f"토큰: {tokens}")

✅ 단어 추가: 리비안 (NNP)
✅ 단어 추가: 테슬라 (NNP)
✅ 단어 추가: 전기차 (NNG)
원문: 리비안은 언제 설립되었나요?
토큰: ['리비안', '은', '언제', '설립', '되', '었', '나요', '?']


#### 4.2.2 BM25 검색기 생성
- uv pip install rank_bm25

In [14]:
# BM25 검색기 생성 (한국어 토크나이저 미적용)
bm25_retriever = BM25Retriever.from_documents(
    documents=processed_docs,
    k=5
)

# 검색 테스트
keyword_results = bm25_retriever.invoke(query)

print(f"Query: {query}")
print(f"🔍 검색 결과 ({len(keyword_results)}개)")
for i, doc in enumerate(keyword_results, 1):
    print(f"\n[{i}] 회사: {doc.metadata.get('company', 'N/A')}")
    print(f"내용:\n{doc.page_content}")
    print("-" * 50)


Query: 리비안은 언제 설립되었나요?
🔍 검색 결과 (5개)

[1] 회사: 리비안(rivian)
내용:
<Document>
**소송**

- 2020년 7월, Tesla는 Rivian이 독점 정보를 훔치고 직원을 빼갔다고 주장하며 소송을 제기했습니다.
- 2021년 3월, Illinois Automobile Dealers Association은 Rivian과 Lucid Motors가 소비자에게 직접 판매했다는 이유로 소송을 제기했습니다.
- 2021년 11월, 전 VP Laura Schwab은 차별 혐의로 소송을 제기하고 차량 가격 책정 및 안전 표준에 대한 우려를 제기했습니다.
</Document>
<Source>이 문서는 '리비안(rivian)'에 대한 문서입니다.</Source>
--------------------------------------------------

[2] 회사: 테슬라(tesla)
내용:
<Document>
2021년 초, Tesla는 Bitcoin에 15억 달러를 투자하고 환경 문제로 인해 중단하기 전에 잠시 결제 수단으로 허용했습니다. 2022년 7월까지 Tesla는 Bitcoin 보유량의 약 75%를 매각했습니다. 2023년 5월과 2024년 2월 사이에 북미 EV 제조업체는 Tesla의 북미 충전 표준으로 전환할 계획을 발표했습니다.

2023년 11월, Tesla는 Cybertruck 배송을 시작했습니다. 2024년 4월, 회사는 직원 10% 감축을 발표하고 6월에 법인 설립지를 델라웨어에서 텍사스로 이전했습니다. 2024년 10월, Tesla는 미래의 차량 호출 서비스인 Tesla Network를 위해 Cybercab 및 Robovan의 컨셉 버전을 공개했습니다.
</Document>
<Source>이 문서는 '테슬라(tesla)'에 대한 문서입니다.</Source>
--------------------------------------------------

[3] 회사: 테슬라(tesla)
내용:
<Document

In [15]:
def create_bm25_retriever(documents, kiwi_model, k=5):
    """
    BM25 검색기 생성 (한국어 토크나이저 적용)
    
    Args:
        documents (list): Document 객체 리스트
        kiwi_model: Kiwi 토크나이저
        k (int): 반환할 문서 수
    
    Returns:
        BM25Retriever: BM25 검색기 객체
    """
    
    # 한국어 토크나이저 적용
    def preprocess_func(text):
        return korean_tokenizer(text, kiwi_model)
    
    # BM25 검색기 생성
    bm25_retriever = BM25Retriever.from_documents(
        documents=documents,
        preprocess_func=preprocess_func,
        k=k
    )
    
    print(f"✅ BM25 검색기 생성 완료: {len(documents)}개 문서 인덱싱")
    return bm25_retriever

# BM25 검색기 생성
bm25_retriever = create_bm25_retriever(processed_docs, kiwi_model, k=5)

✅ BM25 검색기 생성 완료: 39개 문서 인덱싱


In [16]:
# 검색 테스트
keyword_results = bm25_retriever.invoke(query)

print(f"Query: {query}")
print(f"🔍 검색 결과 ({len(keyword_results)}개)")
for i, doc in enumerate(keyword_results, 1):
    print(f"\n[{i}] 회사: {doc.metadata.get('company', 'N/A')}")
    print(f"내용:\n{doc.page_content}")
    print("-" * 50)


Query: 리비안은 언제 설립되었나요?
🔍 검색 결과 (5개)

[1] 회사: 테슬라(tesla)
내용:
<Document>
### SolarCity 및 Model 3 (2016–2018)

Tesla는 2016년 11월 SolarCity를 26억 달러에 인수하여 Tesla Energy를 설립했습니다. 2017년 2월, Tesla Motors는 사명을 Tesla, Inc.로 변경했습니다.

Model 3 세단은 2016년 4월에 공개되어 일주일 만에 325,000건 이상의 예약이 접수되었습니다. "생산 지옥"으로 묘사된 생산 문제로 인해 지연이 발생했습니다. 2018년 말까지 Model 3는 세계에서 가장 많이 팔린 전기 자동차(2018-2021)가 되었습니다. 2018년 8월, Musk는 Tesla를 비상장으로 전환할 계획을 발표했지만 실현되지 않았고 논란과 SEC의 증권 사기 혐의로 이어졌습니다.

### 글로벌 확장 및 Model Y (2019–현재)
</Document>
<Source>이 문서는 '테슬라(tesla)'에 대한 문서입니다.</Source>
--------------------------------------------------

[2] 회사: 테슬라(tesla)
내용:
<Document>
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자

#### 4.2.3 BM25 점수 확인

In [17]:
def analyze_bm25_scores(bm25_retriever, query, kiwi_model, top_k=5):
    """
    BM25 점수 분석
    
    Args:
        bm25_retriever: BM25 검색기
        query (str): 검색 쿼리
        kiwi_model: Kiwi 토크나이저
        top_k (int): 상위 k개 결과
    """
    # 쿼리 토큰화
    tokenized_query = korean_tokenizer(query, kiwi_model)
    print(f"쿼리 토큰: {tokenized_query}")
    
    # BM25 점수 계산
    doc_scores = bm25_retriever.vectorizer.get_scores(tokenized_query)
    
    # 점수 정렬
    doc_scores_sorted = sorted(
        enumerate(doc_scores), 
        key=lambda x: x[1], 
        reverse=True
    )
    
    print(f"\n📊 상위 {top_k}개 문서의 BM25 점수:")
    print("-" * 80)
    
    for rank, (idx, score) in enumerate(doc_scores_sorted[:top_k], 1):
        doc = bm25_retriever.docs[idx]
        print(f"[{rank}] 점수: {score:.4f}")
        print(f"    회사: {doc.metadata.get('company', 'N/A')}")
        print(f"    내용: {doc.page_content[:200]}...")
        print("-" * 80)

# BM25 점수 분석
analyze_bm25_scores(bm25_retriever, query, kiwi_model)

쿼리 토큰: ['리비안', '은', '언제', '설립', '되', '었', '나요', '?']

📊 상위 5개 문서의 BM25 점수:
--------------------------------------------------------------------------------
[1] 점수: 4.3159
    회사: 테슬라(tesla)
    내용: <Document>
### SolarCity 및 Model 3 (2016–2018)

Tesla는 2016년 11월 SolarCity를 26억 달러에 인수하여 Tesla Energy를 설립했습니다. 2017년 2월, Tesla Motors는 사명을 Tesla, Inc.로 변경했습니다.

Model 3 세단은 2016년 4월에 공개되어 일주일 만에 325,0...
--------------------------------------------------------------------------------
[2] 점수: 4.1388
    회사: 테슬라(tesla)
    내용: <Document>
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해...
--------------------------------------------------------------------------------
[3] 점수: 3.9886
    회사: 리비안(rivian)
    내용: <Document>
**EV 충전**

Rivian은 미국과 캐나다 전역에 공공 충전소 네트워크를 개발하고 있습니다. Rivian은 2021년에 Waypoint 충전기 설치를 시작했지만 2023년 4월에 Rivian Adventure Network가 Rivian 

### 4.3 하이브리드 검색 (Hybrid Search)

In [20]:
def create_hybrid_retriever(vector_store, bm25_retriever, weights=None):
    """
    하이브리드 검색기 생성
    
    Args:
        vector_store: 벡터 저장소
        bm25_retriever: BM25 검색기
        weights (list): 가중치 [의미론적, 키워드]
    
    Returns:
        EnsembleRetriever: 하이브리드 검색기
    """
    if weights is None:
        weights = [0.5, 0.5]  # 기본값: 동일한 가중치
    
    # 의미론적 검색기 생성
    semantic_retriever = vector_store.as_retriever(search_kwargs={"k": 5})
    
    # 앙상블 검색기 생성
    ensemble_retriever = EnsembleRetriever(
        retrievers=[semantic_retriever, bm25_retriever],
        weights=weights
    )
    
    print(f"✅ 하이브리드 검색기 생성 완료")
    print(f"   가중치: 의미론적 {weights[0]}, 키워드 {weights[1]}")
    
    return ensemble_retriever

# 하이브리드 검색기 생성
hybrid_retriever = create_hybrid_retriever(vector_store, bm25_retriever)

✅ 하이브리드 검색기 생성 완료
   가중치: 의미론적 0.5, 키워드 0.5


In [21]:
# 검색 테스트
hybrid_results = hybrid_retriever.invoke(query)

print(f"Query: {query}")
print(f"🔍 검색 결과 ({len(hybrid_results)}개)")
for i, doc in enumerate(hybrid_results, 1):
    print(f"\n[{i}] 회사: {doc.metadata.get('company', 'N/A')}")
    print(f"내용:\n{doc.page_content}")
    print("-" * 50)


Query: 리비안은 언제 설립되었나요?
🔍 검색 결과 (9개)

[1] 회사: 리비안(rivian)
내용:
<Document>
Rivian Automotive, Inc.는 2009년에 설립된 미국의 전기 자동차 제조업체, 자동차 기술 및 야외 레크리에이션 회사입니다.

**주요 정보:**
</Document>
<Source>이 문서는 '리비안(rivian)'에 대한 문서입니다.</Source>
--------------------------------------------------

[2] 회사: 리비안(rivian)
내용:
<Document>
- **회사 유형:** 상장
- **거래소:** NASDAQ: RIVN
- **설립:** 2009년 6월, 플로리다 주 록ledge
- **설립자:** R. J. 스캐린지
- **본사:** 미국 캘리포니아 주 어바인
- **서비스 지역:** 북미
- **주요 인물:** R. J. 스캐린지 (CEO)
- **제품:** 전기 자동차, 배터리
- **생산량 (2023):** 57,232대
- **서비스:** 전기 자동차 충전, 자동차 보험
- **수익 (2023):** 44억 3천만 미국 달러
- **순이익 (2023):** -54억 미국 달러
- **총 자산 (2023):** 168억 미국 달러
</Document>
<Source>이 문서는 '리비안(rivian)'에 대한 문서입니다.</Source>
--------------------------------------------------

[3] 회사: 테슬라(tesla)
내용:
<Document>
### SolarCity 및 Model 3 (2016–2018)

Tesla는 2016년 11월 SolarCity를 26억 달러에 인수하여 Tesla Energy를 설립했습니다. 2017년 2월, Tesla Motors는 사명을 Tesla, Inc.로 변경했습니다.

Model 3 세단은 2016년 4월에 공개되어 일주일 만에 325,000건 이상의 예약이 접수되었습니다.

---

## 5. 평가 및 비교

### 5.1 평가 데이터셋 준비

In [22]:
def load_evaluation_dataset(file_path):
    """
    평가 데이터셋 로드
    
    Args:
        file_path (str): 평가 데이터 파일 경로
    
    Returns:
        pandas.DataFrame: 평가 데이터셋
    """
    try:
        if file_path.endswith('.xlsx'):
            df = pd.read_excel(file_path)
        elif file_path.endswith('.csv'):
            df = pd.read_csv(file_path)
        else:
            raise ValueError("지원하지 않는 파일 형식")
        
        print(f"✅ 평가 데이터셋 로드: {len(df)}개 질문")
        return df
    except Exception as e:
        print(f"❌ 데이터셋 로드 실패: {e}")
        return None

# 평가 데이터셋 로드

eval_df = load_evaluation_dataset("./data/synthetic_testset.csv")
eval_df.head(3)

✅ 평가 데이터셋 로드: 50개 질문


,user_input,reference_contexts,reference,synthesizer_name
0,"Nikola Tesla가 Tesla, Inc.의 이름에 어떻게 연결되어 있는지, 그...","['Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회...","Tesla, Inc.는 2003년 7월 Martin Eberhard와 Marc Ta...",single_hop_specifc_query_synthesizer
1,Tesla가 전기차 시장에서 차지하는 위치와 주요 성과에 대해 설명해 주실 수 있나요?,['Tesla의 차량 생산은 2008년 Roadster로 시작하여 Model S (...,"Tesla는 2020년 7월 이후 세계에서 가장 가치 있는 자동차 제조업체이며, 2...",single_hop_specifc_query_synthesizer
2,Martin Eberhard는 Tesla의 설립 과정에서 어떤 역할을 했습니까?,"['Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, M...","Martin Eberhard는 2003년 7월 1일에 Tesla Motors, In...",single_hop_specifc_query_synthesizer


In [23]:
def prepare_evaluation_data(df):
    """
    평가 데이터 전처리
    
    Args:
        df (pandas.DataFrame): 원본 데이터프레임
    
    Returns:
        tuple: (질문 리스트, 정답 문서 리스트)
    """
    questions = df['user_input'].tolist()
    
    # 정답 문서 파싱
    reference_contexts = []
    for contexts in df['reference_contexts']:
        if isinstance(contexts, str):
            # 문자열을 리스트로 변환
            context_list = eval(contexts)
        else:
            context_list = contexts
        
        # Document 객체로 변환
        docs = [Document(page_content=ctx) for ctx in context_list]
        reference_contexts.append(docs)
    
    return questions, reference_contexts

# 평가 데이터 전처리
questions, reference_contexts = prepare_evaluation_data(eval_df)

# 평가 데이터 확인
for i, (q, refs) in enumerate(zip(questions[:3], reference_contexts[:3])):
    print(f"\n[질문 {i+1}]")
    print(f"질문: {q}")
    print(f"정답 문서: {len(refs)}개")
    for j, ref in enumerate(refs):
        print(f"  [{j+1}] 내용: {ref.page_content[:50]}...")  # 내용 일부만 출력
    print("-" * 50)

# 평가 데이터 확인
for i, (q, refs) in enumerate(zip(questions[-3:], reference_contexts[-3:])):
    print(f"\n[질문 {i+1}]")
    print(f"질문: {q}")
    print(f"정답 문서: {len(refs)}개")
    for j, ref in enumerate(refs):
        print(f"  [{j+1}] 내용: {ref.page_content[:50]}...")  # 내용 일부만 출력
    print("-" * 50)



[질문 1]
질문: Nikola Tesla가 Tesla, Inc.의 이름에 어떻게 연결되어 있는지, 그리고 회사 설립 과정에서 어떤 의미를 가지는지 자세히 설명해 주세요.
정답 문서: 1개
  [1] 내용: Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 ...
--------------------------------------------------

[질문 2]
질문: Tesla가 전기차 시장에서 차지하는 위치와 주요 성과에 대해 설명해 주실 수 있나요?
정답 문서: 1개
  [1] 내용: Tesla의 차량 생산은 2008년 Roadster로 시작하여 Model S (2012),...
--------------------------------------------------

[질문 3]
질문: Martin Eberhard는 Tesla의 설립 과정에서 어떤 역할을 했습니까?
정답 문서: 1개
  [1] 내용: Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논...
--------------------------------------------------

[질문 1]
질문: Tesla의 기업 공개(IPO)가 미국 전기차(Electric vehicles) 시장에서 어떤 의미를 가지며, IPO 이전과 이후 Tesla의 자금 조달 및 사업 확장에 어떤 변화가 있었는지 설명해 주시겠습니까?
정답 문서: 2개
  [1] 내용: <1-hop>

Roadster 생산은 2008년에 시작되었습니다. 2009년 1월까지 T...
  [2] 내용: <2-hop>

Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. ...
--------------------------------------------------

[질문 2]
질문: Tesla의 기업 공개(IPO) 시점과 그 당시 출시된 Tesla vehicle models에는 어떤 것들이 있었나요?
정답 문서: 2개
 

### 5.2 ranx-k 라이브러리 활용한 평가

- ROUGE 점수 기반 평가 
- 아이디어: 텍스트 오버랩을 통한 직접적 유사도 측정

- **ranx-k 설치 방법**

    ```bash
    # ranx-k 라이브러리 설치
    uv pip install ranx-k
    ```

- **ranx 이슈 사항**

    - **ID 매칭 의존성**: 정확한 문서 ID 일치가 필요
    - **의미적 유사도 무시**: 내용이 비슷해도 ID가 다르면 0점
    - **청킹 방식 변화 대응 불가**: 청크 크기나 방식이 바뀌면 평가 불가능
    - **실제 사용 환경과 괴리**: 실제로는 의미적 관련성이 중요

In [ ]:
from ranx_k.evaluation import evaluate_with_ranx_similarity

# ranx-k 평가 실행 (rouge 점수가 높은 경우) -> 문자열 유사도 기반 평가
ranx_k_results = evaluate_with_ranx_similarity(
    retriever=hybrid_retriever,
    questions=questions,
    reference_contexts=reference_contexts,
    k=5,
    method='kiwi_rouge',  
    similarity_threshold=0.8,
)

In [ ]:
# ranx-k 평가 실행 (embedding 점수가 높은 경우) -> 의미적 유사도 기반 평가
ranx_k_results = evaluate_with_ranx_similarity(
    retriever=hybrid_retriever,
    questions=questions,
    reference_contexts=reference_contexts,
    k=5,
    method='embedding',  
    embedding_model="BAAI/bge-m3",
    similarity_threshold=0.9
)

### 5.4 검색 방법 비교

In [ ]:
def compare_retrieval_methods(vector_store, bm25_retriever, questions, reference_contexts, k=5):
    """
    다양한 검색 방법 성능 비교
    
    Args:
        vector_store: 벡터 저장소
        bm25_retriever: BM25 검색기
        questions (list): 질문 리스트
        reference_contexts (list): 정답 문서 리스트
        k (int): 평가할 상위 k개 결과
    
    Returns:
        pandas.DataFrame: 비교 결과
    """
    results = []
    
    # 검색 방법들 정의
    bm25_retriever.k = k
    retrievers = {
        "의미론적 검색": vector_store.as_retriever(search_kwargs={"k": k}),
        "키워드 검색": bm25_retriever,
        "하이브리드 (5:5)": create_hybrid_retriever(vector_store, bm25_retriever, [0.5, 0.5]),
        "하이브리드 (7:3)": create_hybrid_retriever(vector_store, bm25_retriever, [0.7, 0.3]),
        "하이브리드 (3:7)": create_hybrid_retriever(vector_store, bm25_retriever, [0.3, 0.7]),
    }
    
    for method_name, retriever in retrievers.items():
        print(f"🔄 {method_name} 평가 중...")
        
        # ranx-k 평가 실행 (rouge 점수가 높은 경우) -> 문자열 유사도 기반 평가
        ranx_k_results = evaluate_with_ranx_similarity(
            retriever=retriever,
            questions=questions,
            reference_contexts=reference_contexts,
            k=k,
            method='kiwi_rouge',  
            similarity_threshold=0.8,
        )
        
        # 평가 결과 정리 
        result = {
            "method": method_name,
            "hit_rate": ranx_k_results.get("hit_rate@5", 0),
            "ndcg": ranx_k_results.get("ndcg@5", 0),
            "map": ranx_k_results.get("map@5", 0),
            "mrr": ranx_k_results.get("mrr", 0),
        }

        results.append(result)

    # 결과 DataFrame 생성
    results_df = pd.DataFrame(results)
    results_df.set_index("method", inplace=True)
    results_df.sort_values(by="hit_rate", ascending=False, inplace=True)

    return results_df


# 검색 방법 성능 비교
comparison_results = compare_retrieval_methods(vector_store, bm25_retriever, questions, reference_contexts, k=5)

In [ ]:
comparison_results

### 5.5 시각화

In [ ]:
def visualize_comparison_results(comparison_df):
    """
    검색 방법 비교 결과 시각화
    
    Args:
        comparison_df (pandas.DataFrame): 비교 결과 데이터프레임

        result = {
            "method": method_name,
            "hit_rate": ranx_k_results.get("hit_rate@5", 0),
            "ndcg": ranx_k_results.get("ndcg@5", 0),
            "map": ranx_k_results.get("map@5", 0),
            "mrr": ranx_k_results.get("mrr", 0),
        }

    """
    
    # 그래프 생성
    fig, ax = plt.subplots(1, 4, figsize=(12, 8))
    sns.barplot(x=comparison_df.index, y='hit_rate', data=comparison_df, ax=ax[0])
    ax[0].set_title('Hit Rate@5')
    ax[0].set_ylabel('Hit Rate')
    ax[0].set_xlabel('Retrieval Method')        
    ax[0].tick_params(axis='x', rotation=45)

    sns.barplot(x=comparison_df.index, y='ndcg', data=comparison_df, ax=ax[1])
    ax[1].set_title('NDCG@5')
    ax[1].set_ylabel('NDCG')
    ax[1].set_xlabel('Retrieval Method')
    ax[1].tick_params(axis='x', rotation=45)                

    sns.barplot(x=comparison_df.index, y='map', data=comparison_df, ax=ax[2])
    ax[2].set_title('MAP@5')
    ax[2].set_ylabel('MAP')         
    ax[2].set_xlabel('Retrieval Method')
    ax[2].tick_params(axis='x', rotation=45)    

    sns.barplot(x=comparison_df.index, y='mrr', data=comparison_df, ax=ax[3])
    ax[3].set_title('MRR')
    ax[3].set_ylabel('MRR')
    ax[3].set_xlabel('Retrieval Method')
    ax[3].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    # 그래프 출력
    plt.savefig("retrieval_comparison_results.png")

# 검색 방법 비교 결과 시각화
visualize_comparison_results(comparison_results)

---

## 6. 실습 문제

**목표**: 기본적인 하이브리드 검색 시스템을 구축하고 테스트해보세요.

**문제**:
1. 제공된 텍스트 데이터를 사용하여 벡터 저장소와 BM25 검색기를 생성하세요.
2. 동일한 가중치(0.5, 0.5)로 하이브리드 검색기를 만드세요.
3. 다음 질문들에 대해 검색을 수행하고 결과를 비교하세요:
   - "테슬라의 전기차 모델은 어떤 것들이 있나요?"
   - "리비안의 경쟁력은 무엇인가요?"
   - "테슬라와 리비안이 경쟁하는 분야는 무엇인가요?"

In [ ]:
# 여기에 코드를 작성하세요

# 1. 데이터 로드 및 전처리

# 2. 벡터 저장소 생성

# 3. BM25 검색기 생성

# 4. 하이브리드 검색기 생성

# 5. 검색 테스트
queries = [
    "테슬라의 전기차 모델은 어떤 것들이 있나요?",
    "리비안의 경쟁력은 무엇인가요?",
    "테슬라와 리비안이 경쟁하는 분야는 무엇인가요?"
]

for query in queries:
    print(f"질문: {query}")
    # 검색 수행 및 결과 출력
    pass